# 00 · Setup, data, and evaluation pipeline (Charades-STA)

**Kaggle settings for this notebook:** Accelerator = **None** (this step is CPU-only, so save your GPU quota), Internet = **On**, Persistence = **Files only**.

What this notebook does:
1. Gets the code and the annotations.
2. Builds clean `test` and `val` query lists.
3. Downloads only the videos we need (about 1,600 of the 9,848 Charades videos).
4. Verifies every video decodes.
5. Runs oracle, random, and prior baselines to prove the evaluation code is correct.

Everything it produces lands in `/kaggle/working/`, which you then turn into a Kaggle Dataset that every later notebook reuses.

## 1. Install and get the code

In [ ]:
!pip install -q remotezip

# Replace with your team's repo URL after you push the starter code to GitHub.
REPO_URL = "https://github.com/YOUR_USERNAME/ai-video-investigator.git"
!rm -rf /tmp/repo && git clone -q $REPO_URL /tmp/repo

import sys
sys.path.insert(0, "/tmp/repo")
!python /tmp/repo/tests/test_basics.py

## 2. Paths

In [ ]:
from pathlib import Path
WORK    = Path("/kaggle/working")
ANN_DIR = WORK / "annotations"   # cleaned query lists (jsonl)
VID_DIR = WORK / "videos"        # mp4 files we actually use
RES_DIR = WORK / "results"       # results.csv experiment log
RAW     = Path("/tmp/raw")       # scratch space, not saved
for d in (ANN_DIR, VID_DIR, RES_DIR, RAW):
    d.mkdir(parents=True, exist_ok=True)

QUICK = False        # True = only 100 test videos, for a fast dry run
N_VAL_VIDEOS = 300   # held-out train videos used for tuning

## 3. Download annotations

- **Video lengths** come from the official Charades annotation zip (a few MB).
- **Charades-STA query files** (`charades_sta_train.txt` and `charades_sta_test.txt`) come from the DRN GitHub repo.

If the DRN step can't find the files, download them manually (they are linked from the TALL and DRN repos), upload them as a small Kaggle Dataset, and set `STA_TRAIN` and `STA_TEST` by hand.

In [ ]:
from vidsearch.download import CHARADES_ANN_URL
!wget -q -O /tmp/raw/Charades.zip $CHARADES_ANN_URL && unzip -q -o /tmp/raw/Charades.zip -d /tmp/raw
!git clone -q --depth 1 https://github.com/Alvin-Zeng/DRN.git /tmp/raw/DRN || echo "DRN clone failed"

CSV_TRAIN = next(RAW.rglob("Charades_v1_train.csv"))
CSV_TEST  = next(RAW.rglob("Charades_v1_test.csv"))
sta = [p for p in RAW.rglob("*.txt") if "sta" in p.name.lower()]
print("Found STA candidates:", sta)
STA_TRAIN = next(p for p in sta if "train" in p.name.lower())
STA_TEST  = next(p for p in sta if "test"  in p.name.lower())
print(CSV_TRAIN, CSV_TEST, STA_TRAIN, STA_TEST, sep="\n")
!head -3 "$STA_TEST"

## 4. Parse, clean, and split

In [ ]:
from vidsearch.data import parse_sta, load_durations, attach_durations, make_val_split, save_jsonl

durations = load_durations(CSV_TRAIN, CSV_TEST)
train, d_tr = attach_durations(parse_sta(STA_TRAIN), durations, "train")
test,  d_te = attach_durations(parse_sta(STA_TEST),  durations, "test")
val = make_val_split(train, n_val_videos=N_VAL_VIDEOS, seed=0)

if QUICK:
    keep = sorted({r["vid"] for r in test})[:100]
    test = [r for r in test if r["vid"] in keep]

for name, recs, dropped in [("train", train, d_tr), ("val", val, 0), ("test", test, d_te)]:
    print(f"{name:5s}: {len(recs):5d} queries, {len({r['vid'] for r in recs}):5d} videos, dropped {dropped}")

save_jsonl(train, ANN_DIR / "train.jsonl")   # used only for the prior baseline and stats
save_jsonl(val,   ANN_DIR / "val.jsonl")
save_jsonl(test,  ANN_DIR / "test.jsonl")
print(test[0])

## 5. Dataset statistics (reuse these plots in your report)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
gt_len = np.array([r["end"] - r["start"] for r in test])
dur    = np.array([r["duration"] for r in {r["vid"]: r for r in test}.values()])
nstart = np.array([r["start"] / r["duration"] for r in test])
nend   = np.array([r["end"]   / r["duration"] for r in test])
words  = np.array([len(r["query"].split()) for r in test])

fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
ax[0].hist(dur, 40);    ax[0].set_title("Video duration (s)")
ax[1].hist(gt_len, 40); ax[1].set_title("GT moment length (s)")
ax[2].hist(nstart, 30, alpha=.6, label="start"); ax[2].hist(nend, 30, alpha=.6, label="end")
ax[2].legend(); ax[2].set_title("Normalized GT position")
ax[3].hist(words, range(1, 20)); ax[3].set_title("Query length (words)")
plt.tight_layout(); plt.savefig(RES_DIR / "dataset_stats.png", dpi=120); plt.show()

print(f"median video {np.median(dur):.1f}s | median moment {np.median(gt_len):.1f}s "
      f"| moments < 3s: {(gt_len < 3).mean():.1%} | moment/video ratio {np.median(gt_len / [r['duration'] for r in test]):.2f}")

## 6. Download only the videos we need

This uses HTTP range requests, so it fetches individual mp4s out of the large remote zip without downloading the whole archive. It is **resumable**: if the session dies, re-run this cell and it skips videos already on disk.

Expect roughly 2–3 GB and about 15–40 minutes.

In [ ]:
from vidsearch.download import download_subset
need = sorted({r["vid"] for r in test} | {r["vid"] for r in val})
print("videos needed:", len(need))
report = download_subset(need, VID_DIR)
print({k: (v if not isinstance(v, list) else len(v)) for k, v in report.items()})
!du -sh /kaggle/working/videos

## 7. Verify every video decodes and matches its annotated length

In [ ]:
from vidsearch.video_io import video_meta
from tqdm.auto import tqdm
bad, mismatch = [], []
for v in tqdm(need):
    m = video_meta(VID_DIR / f"{v}.mp4")
    if m is None or m["n_frames"] == 0:
        bad.append(v)
    elif abs(m["duration"] - durations[v]) > 2.0:
        mismatch.append((v, round(m["duration"], 1), durations[v]))
print("unreadable:", len(bad), bad[:10])
print("duration mismatch > 2s:", len(mismatch), mismatch[:5])

# Drop queries whose video is unreadable so every later notebook sees a clean list.
if bad:
    bad_set = set(bad)
    test = [r for r in test if r["vid"] not in bad_set]
    val  = [r for r in val  if r["vid"] not in bad_set]
    save_jsonl(val, ANN_DIR / "val.jsonl"); save_jsonl(test, ANN_DIR / "test.jsonl")
    print("re-saved; test queries:", len(test))

## 8. Look at one example (frames at 1 fps; red border = inside the ground-truth moment)

In [ ]:
from vidsearch.video_io import sample_frames
r = test[0]
frames, times = sample_frames(VID_DIR / f"{r['vid']}.mp4", fps=1.0, max_side=320)
idx = np.linspace(0, len(frames) - 1, 12).astype(int)
fig, ax = plt.subplots(2, 6, figsize=(18, 5))
for a, i in zip(ax.flat, idx):
    a.imshow(frames[i]); a.set_title(f"t={times[i]:.1f}s"); a.set_xticks([]); a.set_yticks([])
    if r["start"] <= times[i] <= r["end"]:
        for s in a.spines.values(): s.set_color("red"); s.set_linewidth(4)
fig.suptitle(f'"{r["query"]}"  GT [{r["start"]:.1f}, {r["end"]:.1f}]s of {r["duration"]:.1f}s')
plt.tight_layout(); plt.show()
print("frames:", frames.shape)

## 9. Sanity-check the evaluation

Expected outcomes:
- **Oracle** must score exactly 100 on every metric.
- **Random** should score low.
- **Prior** (always predicting the average GT position) is the bar your methods must clearly beat. Report it, because it reveals dataset bias.

In [ ]:
from vidsearch.metrics import evaluate
from vidsearch.baselines import oracle_predictions, random_predictions, prior_predictions
from vidsearch.utils import log_result
import pandas as pd

rows = {}
for name, preds in [("oracle", oracle_predictions(test)),
                    ("random", random_predictions(test, k=5)),
                    ("prior",  prior_predictions(train, test))]:
    summary, _ = evaluate(preds, test)
    rows[name] = summary
    if name != "oracle":
        log_result(RES_DIR / "results.csv", name, "test", summary)
assert rows["oracle"]["R1@0.7"] == 100, "evaluation bug!"
pd.DataFrame(rows).T.round(1)

## 10. Save everything as a reusable Kaggle Dataset

1. Click **Save Version**, choose **Save & Run All (Commit)**, and wait for it to finish (about the same time as running this notebook).
2. Open the saved version, go to the **Output** tab, and click **New Dataset**. Name it `charades-sta-vidsearch`.
3. In every later notebook, use **Add Input** to attach that dataset. It appears under `/kaggle/input/charades-sta-vidsearch/` with `videos/`, `annotations/`, and `results/` inside.

Next notebook: `01_frame_baseline`, which extracts SigLIP features once (on GPU) and builds the frame-level baseline.